# 02 - Exploratory Data Analysis

This notebook explores the synthetic epidemiological surveillance dataset generated in the previous step.

The goal is to understand the distribution of the main variables, inspect temporal and regional patterns, and identify relationships associated with future outbreak risk.

## 1. Objectives

This exploratory analysis focuses on:

- understanding the structure of the dataset;
- checking outbreak and non-outbreak class balance;
- exploring temporal trends in reported cases;
- comparing regional disease burden;
- studying epidemiological, environmental and mobility-related variables;
- identifying possible predictors of 7-day outbreak risk.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)


## 2. Load Dataset

In [ ]:
DATA_PATH = Path("data/raw/epiwatch_epidemiological_surveillance_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Run src/simulate_data.py before executing this notebook."
    )

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])

df.head()

## 3. Dataset Overview

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"Regions: {df['region'].nunique()}")
print(f"Start date: {df['date'].min().date()}")
print(f"End date: {df['date'].max().date()}")

In [ ]:
df.info()

## 4. Missing Values

A first quality-control step is checking whether the generated dataset contains missing values after temporal feature engineering.

In [ ]:
missing_values = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_values[missing_values > 0]

## 5. Target Distribution

The target variable is `outbreak_7d`, which indicates whether a region is expected to experience outbreak risk within the next seven days.

In [ ]:
target_counts = df["outbreak_7d"].value_counts().sort_index()
target_proportions = df["outbreak_7d"].value_counts(normalize=True).sort_index()

pd.DataFrame({
    "count": target_counts,
    "proportion": target_proportions,
})

In [ ]:
plt.figure(figsize=(6, 4))
target_counts.plot(kind="bar")

plt.title("Distribution of 7-Day Outbreak Risk Classes")
plt.xlabel("Outbreak within 7 days")
plt.ylabel("Number of observations")
plt.xticks(
    ticks=[0, 1],
    labels=["No outbreak", "Outbreak"],
    rotation=0,
)

plt.tight_layout()
plt.show()

## 6. Descriptive Statistics

The following table summarizes the main numerical variables in the dataset.

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns

df[numeric_columns].describe().T

## 7. Reported Cases Over Time

Aggregating reported cases over time helps identify simulated epidemic waves and global temporal patterns.

In [ ]:
daily_cases = (
    df.groupby("date")["reported_cases"]
    .sum()
    .reset_index()
)

plt.figure(figsize=(12, 5))
plt.plot(daily_cases["date"], daily_cases["reported_cases"])

plt.title("Total Reported Cases Over Time")
plt.xlabel("Date")
plt.ylabel("Reported cases")

plt.tight_layout()
plt.show()

## 8. Regional Disease Burden

This section compares the cumulative reported cases across regions.

In [ ]:
regional_cases = (
    df.groupby("region")["reported_cases"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(11, 5))
regional_cases.plot(kind="bar")

plt.title("Total Reported Cases by Region")
plt.xlabel("Region")
plt.ylabel("Total reported cases")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

## 9. Regional Incidence

Incidence per 100,000 inhabitants allows comparison across regions with different population sizes.

In [ ]:
regional_incidence = (
    df.groupby("region")["incidence_per_100k"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(11, 5))
regional_incidence.plot(kind="bar")

plt.title("Average Incidence per 100,000 by Region")
plt.xlabel("Region")
plt.ylabel("Average incidence per 100,000")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

## 10. Highest-Burden Regional Time Series

The plot below shows reported case trajectories for the five regions with the highest cumulative number of reported cases.

In [ ]:
top_regions = regional_cases.head(5).index

plt.figure(figsize=(12, 6))

for region in top_regions:
    subset = df[df["region"] == region]
    plt.plot(subset["date"], subset["reported_cases"], label=region)

plt.title("Reported Cases Over Time in Highest-Burden Regions")
plt.xlabel("Date")
plt.ylabel("Reported cases")
plt.legend(title="Region")

plt.tight_layout()
plt.show()

## 11. Outbreak Risk by Region

This analysis estimates the proportion of observations classified as future outbreak risk for each region.

In [ ]:
regional_outbreak_rate = (
    df.groupby("region")["outbreak_7d"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(11, 5))
regional_outbreak_rate.plot(kind="bar")

plt.title("Proportion of Future Outbreak Risk by Region")
plt.xlabel("Region")
plt.ylabel("Outbreak risk proportion")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

## 12. Epidemiological Variables by Target Class

Comparing variables between outbreak and non-outbreak observations helps identify relevant predictors.

In [ ]:
features_to_compare = [
    "reported_cases",
    "Rt",
    "effective_R",
    "spatial_pressure",
    "imported_cases",
    "incidence_per_100k",
    "healthcare_pressure",
    "mobility_index",
    "vaccination_rate",
]

class_summary = (
    df.groupby("outbreak_7d")[features_to_compare]
    .mean()
    .T
)

class_summary.columns = ["No outbreak", "Outbreak"]
class_summary

In [ ]:
selected_features = [
    "reported_cases",
    "effective_R",
    "spatial_pressure",
    "incidence_per_100k",
    "healthcare_pressure",
]

for feature in selected_features:
    plt.figure(figsize=(7, 4))
    df.boxplot(column=feature, by="outbreak_7d")
    plt.title(f"{feature} by Future Outbreak Class")
    plt.suptitle("")
    plt.xlabel("Outbreak within 7 days")
    plt.ylabel(feature)
    plt.xticks([1, 2], ["No outbreak", "Outbreak"])
    plt.tight_layout()
    plt.show()

## 13. Environmental Patterns

The simulation includes temperature, humidity and rainfall as environmental modifiers of transmission.

In [ ]:
environmental_features = ["temperature", "humidity", "rainfall"]

environment_summary = (
    df.groupby("outbreak_7d")[environmental_features]
    .mean()
    .T
)

environment_summary.columns = ["No outbreak", "Outbreak"]
environment_summary

In [ ]:
for feature in environmental_features:
    plt.figure(figsize=(7, 4))
    df.boxplot(column=feature, by="outbreak_7d")
    plt.title(f"{feature} by Future Outbreak Class")
    plt.suptitle("")
    plt.xlabel("Outbreak within 7 days")
    plt.ylabel(feature)
    plt.xticks([1, 2], ["No outbreak", "Outbreak"])
    plt.tight_layout()
    plt.show()

## 14. Correlation Analysis

Correlation analysis provides a first approximation of linear associations between numerical variables and future outbreak risk.

In [ ]:
correlation_features = [
    "reported_cases",
    "Rt",
    "effective_R",
    "spatial_pressure",
    "imported_cases",
    "temperature",
    "humidity",
    "rainfall",
    "mobility_index",
    "vaccination_rate",
    "testing_rate",
    "healthcare_pressure",
    "incidence_per_100k",
    "outbreak_7d",
]

corr = df[correlation_features].corr()

plt.figure(figsize=(11, 9))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation coefficient")

plt.xticks(range(len(correlation_features)), correlation_features, rotation=45, ha="right")
plt.yticks(range(len(correlation_features)), correlation_features)

plt.title("Correlation Matrix of Main Features")
plt.tight_layout()
plt.show()

In [ ]:
corr["outbreak_7d"].sort_values(ascending=False)

## 15. Temporal Feature Inspection

Lagged and rolling features are important in outbreak prediction because recent trends may contain early-warning signals.

In [ ]:
temporal_features = [
    "reported_cases",
    "reported_cases_lag_1",
    "reported_cases_lag_7",
    "reported_cases_rolling_mean_7",
    "growth_rate_7d",
    "outbreak_7d",
]

df[temporal_features].head(10)

In [ ]:
temporal_summary = (
    df.groupby("outbreak_7d")[[
        "reported_cases_lag_1",
        "reported_cases_lag_7",
        "reported_cases_rolling_mean_7",
        "growth_rate_7d",
    ]]
    .mean()
    .T
)

temporal_summary.columns = ["No outbreak", "Outbreak"]
temporal_summary

## 16. Key Findings

This exploratory analysis suggests that future outbreak risk is associated with a combination of:

- recent reported case burden;
- incidence per 100,000 inhabitants;
- spatial pressure from connected regions;
- effective transmission dynamics;
- healthcare pressure;
- temporal lag and rolling features.

These patterns support the use of supervised machine learning models for early outbreak prediction.

## 17. Next Step

The next notebook trains and compares machine learning models using a temporal train/test split to avoid information leakage from future observations.